In [1]:
import os
import time
import cv2
import numpy as np
import mediapipe as mp

from scipy.spatial import distance

import tensorflow as tf
from tensorflow.keras.models import load_model

print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)

TensorFlow: 2.21.0
OpenCV: 4.10.0
MediaPipe: 0.10.35


In [2]:
# Face Recognition Model
face_model = load_model(
    "models/face_recognition_model.keras"
)

# Emotion Model
emotion_model = load_model(
    "models/emotion_recognition/emotion_model.keras"
)

print("Models Loaded Successfully!")

Models Loaded Successfully!


In [3]:
emotion_labels = [
    "Angry",
    "Disgust",
    "Fear",
    "Happy",
    "Neutral",
    "Sad",
    "Surprise"
]

print(emotion_labels)

['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']


In [4]:
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

print(face_detector.empty())

False


In [5]:
BaseOptions = mp.tasks.BaseOptions

FaceLandmarker = mp.tasks.vision.FaceLandmarker

FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions

VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="face_landmarker.task"
    ),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

print("Face Landmarker Ready!")

Face Landmarker Ready!


In [6]:
LEFT_EYE = [33,160,158,133,153,144]

RIGHT_EYE = [362,385,387,263,373,380]


def eye_aspect_ratio(eye):

    A = distance.euclidean(eye[1], eye[5])

    B = distance.euclidean(eye[2], eye[4])

    C = distance.euclidean(eye[0], eye[3])

    return (A+B)/(2.0*C)


EAR_THRESHOLD = 0.22

CONSEC_FRAMES = 3

print("Blink Detection Ready!")

Blink Detection Ready!


In [7]:
print(face_model)

print(emotion_model)

print(face_detector)

print(landmarker)

print(LEFT_EYE)

print(RIGHT_EYE)

<Functional name=functional, built=True>
<Functional name=functional, built=True>
< cv2.CascadeClassifier 000001E12B30B6D0>
[33, 160, 158, 133, 153, 144]
[362, 385, 387, 263, 373, 380]


In [8]:
import cv2

cap = cv2.VideoCapture(0)

print("Camera Open:", cap.isOpened())

ret, frame = cap.read()

print("Frame Captured:", ret)

cap.release()

Camera Open: True
Frame Captured: True


In [9]:
import cv2

cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    print("Faces:", len(faces))

    for (x,y,w,h) in faces:
        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)

    cv2.imshow("Face Detection",frame)

    if cv2.waitKey(1)==ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
F

In [10]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_detector.detectMultiScale(gray,1.3,5)

    for (x,y,w,h) in faces:

        face = frame[y:y+h,x:x+w]

        face = cv2.resize(face,(224,224))
        face = face.astype("float32")/255.0
        face = np.expand_dims(face,0)

        pred = face_model.predict(face,verbose=0)

        print(pred)

        cv2.putText(
            frame,
            "Disha",
            (x,y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0,255,0),
            2
        )

        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)

    cv2.imshow("Recognition",frame)

    if cv2.waitKey(1)==ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

C:\Users\disha\anaconda3\envs\engagement\Lib\site-packages\keras\src\ops\nn.py:959: UserWarning: You are using a softmax over axis -1 of a tensor of shape (1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]
[[1.]]


In [11]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    gray = cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)

    faces = face_detector.detectMultiScale(gray,1.3,5)

    for (x,y,w,h) in faces:

        face = frame[y:y+h,x:x+w]

        face = cv2.resize(face,(224,224))
        face = face.astype("float32")/255.0
        face = np.expand_dims(face,0)

        pred = emotion_model.predict(face,verbose=0)

        emotion = emotion_labels[np.argmax(pred)]

        cv2.putText(
            frame,
            emotion,
            (x,y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255,0,0),
            2
        )

        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)

    cv2.imshow("Emotion",frame)

    if cv2.waitKey(1)==ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [12]:
import cv2
import mediapipe as mp
import time

cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    timestamp = int(time.time()*1000)

    result = landmarker.detect_for_video(
        mp_image,
        timestamp
    )

    print("Faces:", len(result.face_landmarks))

    cv2.imshow("MediaPipe",frame)

    if cv2.waitKey(1)==ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
Faces: 1
F

In [13]:
import cv2
import mediapipe as mp
import time

cap = cv2.VideoCapture(0)

blink_counter = 0
blink_total = 0

EAR_THRESHOLD = 0.22
CONSEC_FRAMES = 3

while True:

    ret, frame = cap.read()

    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    timestamp = int(time.time() * 1000)

    result = landmarker.detect_for_video(mp_image, timestamp)

    if result.face_landmarks:

        landmarks = result.face_landmarks[0]

        left_eye = [(landmarks[i].x, landmarks[i].y) for i in LEFT_EYE]
        right_eye = [(landmarks[i].x, landmarks[i].y) for i in RIGHT_EYE]

        leftEAR = eye_aspect_ratio(left_eye)
        rightEAR = eye_aspect_ratio(right_eye)

        ear = (leftEAR + rightEAR) / 2.0

        cv2.putText(
            frame,
            f"EAR: {ear:.2f}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 0),
            2
        )

        if ear < EAR_THRESHOLD:
            blink_counter += 1

        else:
            if blink_counter >= CONSEC_FRAMES:
                blink_total += 1

            blink_counter = 0

        cv2.putText(
            frame,
            f"Blinks: {blink_total}",
            (20, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2
        )

    cv2.imshow("Blink Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [14]:
# Nose
NOSE = 1

# Chin
CHIN = 152

# Left eye outer corner
LEFT_EYE_OUTER = 33

# Right eye outer corner
RIGHT_EYE_OUTER = 263

# Left mouth corner
LEFT_MOUTH = 61

# Right mouth corner
RIGHT_MOUTH = 291

print("Head Pose Landmark Indices Loaded!")

Head Pose Landmark Indices Loaded!


In [15]:
print(NOSE)
print(CHIN)
print(LEFT_EYE_OUTER)
print(RIGHT_EYE_OUTER)
print(LEFT_MOUTH)
print(RIGHT_MOUTH)

1
152
33
263
61
291


In [16]:
import cv2
import mediapipe as mp
import numpy as np
import time

cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    h, w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    timestamp = int(time.time() * 1000)

    result = landmarker.detect_for_video(mp_image, timestamp)

    if result.face_landmarks:

        face = result.face_landmarks[0]

        face_2d = []
        face_3d = []

        landmark_ids = [
            NOSE,
            CHIN,
            LEFT_EYE_OUTER,
            RIGHT_EYE_OUTER,
            LEFT_MOUTH,
            RIGHT_MOUTH
        ]

        for idx in landmark_ids:

            x = int(face[idx].x * w)
            y = int(face[idx].y * h)

            face_2d.append([x, y])
            face_3d.append([x, y, face[idx].z])

        face_2d = np.array(face_2d, dtype=np.float64)
        face_3d = np.array(face_3d, dtype=np.float64)

        focal_length = w

        cam_matrix = np.array([
            [focal_length, 0, w / 2],
            [0, focal_length, h / 2],
            [0, 0, 1]
        ])

        dist_matrix = np.zeros((4, 1), dtype=np.float64)

        success, rot_vec, trans_vec = cv2.solvePnP(
            face_3d,
            face_2d,
            cam_matrix,
            dist_matrix
        )

        rmat, _ = cv2.Rodrigues(rot_vec)

        angles, _, _, _, _, _ = cv2.RQDecomp3x3(rmat)

        x_angle = angles[0] * 360
        y_angle = angles[1] * 360

        if y_angle < -10:
            direction = "Looking Left"

        elif y_angle > 10:
            direction = "Looking Right"

        elif x_angle < -10:
            direction = "Looking Down"

        elif x_angle > 10:
            direction = "Looking Up"

        else:
            direction = "Looking Forward"

        cv2.putText(
            frame,
            direction,
            (20,40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0,255,0),
            2
        )

    cv2.imshow("Head Pose", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [17]:
# Iris landmark indices
LEFT_IRIS = [468, 469, 470, 471]
RIGHT_IRIS = [473, 474, 475, 476]

print("Gaze Estimation Ready!")

Gaze Estimation Ready!


In [18]:
print(LEFT_IRIS)
print(RIGHT_IRIS)

[468, 469, 470, 471]
[473, 474, 475, 476]


In [19]:
def iris_center(landmarks, iris_indices):
    xs = [landmarks[i].x for i in iris_indices]
    ys = [landmarks[i].y for i in iris_indices]

    return np.mean(xs), np.mean(ys)

print("Iris Function Ready!")

Iris Function Ready!


In [20]:
print(LEFT_IRIS)
print(RIGHT_IRIS)
print(iris_center)

[468, 469, 470, 471]
[473, 474, 475, 476]
<function iris_center at 0x000001E13CDBDDA0>
